# NetraEdge - Face Recognition Training
## MobileFaceNet | GPU: T4 | Time: ~2 hours
## Output: face_recognition.onnx

In [ ]:
#@title Step 1: Check GPU
import torch
print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name())
else:
    print('No GPU! Runtime > Change runtime type > GPU')

In [ ]:
#@title Step 2: Install deps
!pip install -q onnx tqdm scikit-learn
print('Done')

In [ ]:
#@title Step 3: Load LFW via sklearn
from sklearn.datasets import fetch_lfw_people
import os

lfw = fetch_lfw_people(min_faces_per_person=20, resize=0.5, color=True)
print('Images:', lfw.images.shape)
print('Identities:', len(lfw.target_names))
print('Names:', list(lfw.target_names[:10]))

# Save to disk for DataLoader
output_dir = 'lfw_persons'
os.makedirs(output_dir, exist_ok=True)
from PIL import Image
import numpy as np

for i, name in enumerate(lfw.target_names):
    mask = lfw.target == i
    if mask.sum() < 10:
        continue
    person_dir = os.path.join(output_dir, name.replace(' ', '_'))
    os.makedirs(person_dir, exist_ok=True)
    for j, idx in enumerate(np.where(mask)[0]):
        img = Image.fromarray(lfw.images[idx].astype(np.uint8))
        img = img.resize((112, 112), Image.BILINEAR)
        img.save(os.path.join(person_dir, f'{j:04d}.jpg'))

persons = [d for d in os.listdir(output_dir) if os.path.isdir(os.path.join(output_dir, d))]
print('Saved', len(persons), 'persons to', output_dir)

In [ ]:
#@title Step 4: MobileFaceNet Architecture
import torch
import torch.nn as nn
import torch.nn.functional as F

class DWSep(nn.Module):
    def __init__(self, ic, oc, st=1):
        super().__init__()
        self.dw = nn.Conv2d(ic, ic, 3, st, 1, groups=ic, bias=False)
        self.b1 = nn.BatchNorm2d(ic)
        self.pw = nn.Conv2d(ic, oc, 1, bias=False)
        self.b2 = nn.BatchNorm2d(oc)
    def forward(self, x):
        return F.relu(self.b2(self.pw(F.relu(self.b1(self.dw(x))))))

class SE(nn.Module):
    def __init__(self, ch, r=4):
        super().__init__()
        m = max(ch // r, 8)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, m), nn.ReLU(True),
            nn.Linear(m, ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.se(x).unsqueeze(-1).unsqueeze(-1)

class MB(nn.Module):
    def __init__(self, ic, oc, st=1):
        super().__init__()
        m = ic * 2
        self.ex = nn.Sequential(
            nn.Conv2d(ic, m, 1, bias=False), nn.BatchNorm2d(m), nn.ReLU(True))
        self.dw = DWSep(m, oc, st)
        self.se = SE(oc)
        self.res = (st == 1 and ic == oc)
    def forward(self, x):
        o = self.se(self.dw(self.ex(x)))
        return o + x if self.res else o

class MobileFaceNet(nn.Module):
    def __init__(self, ed=128):
        super().__init__()
        self.ed = ed
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 2, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True))
        self.blk = nn.Sequential(
            MB(64, 64), MB(64, 128, 2), MB(128, 128),
            MB(128, 256, 2), MB(256, 256), MB(256, 256),
            MB(256, 512, 2), MB(512, 512), MB(512, 512))
        self.fin = nn.Sequential(
            nn.Conv2d(512, 512, 3, groups=512, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, ed, 1, bias=False), nn.BatchNorm2d(ed))
    def forward(self, x):
        x = self.fin(self.blk(self.stem(x)))
        return F.normalize(x.view(x.size(0), -1), p=2, dim=1)

m = MobileFaceNet(128)
nparams = sum(p.numel() for p in m.parameters())
print('Params:', f'{nparams:,}')
print('Output:', m(torch.randn(2, 3, 112, 112)).shape)

In [ ]:
#@title Step 5: Dataset + Loader
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np, random, glob

class LFW(Dataset):
    def __init__(self, root='lfw_persons', mx=100, aug=True):
        self.samples = []
        self.aug = aug
        ids = sorted([d for d in os.listdir(root) if os.path.isdir(root + '/' + d)])[:mx]
        for i, identity in enumerate(ids):
            for p in glob.glob(root + '/' + identity + '/*.jpg'):
                self.samples.append((p, i))
        print(str(len(self.samples)) + ' images, ' + str(len(ids)) + ' identities')
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        p, l = self.samples[i]
        img = Image.open(p).convert('RGB').resize((112, 112))
        a = np.array(img).astype(np.float32) / 255.0
        if self.aug:
            if random.random() > 0.5:
                a = np.clip(a + np.random.uniform(-0.12, 0.12), 0, 1)
            if random.random() > 0.5:
                a = np.flip(a, axis=1).copy()
        t = torch.from_numpy(a).permute(2, 0, 1).float()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        return (t - mean) / std, l

tr = LFW('lfw_persons', 80)
va = LFW('lfw_persons', 100, aug=False)
trl = DataLoader(tr, 64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(va, 64, shuffle=False, num_workers=2)
print('Train:', len(tr), 'Val:', len(va))

In [ ]:
#@title Step 6: Train (10 epochs)
import time
from tqdm import tqdm

D = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
m = MobileFaceNet(128).to(D)
criterion = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10, eta_min=1e-6)
best = 0

for ep in range(10):
    t0 = time.time()
    m.train()
    tc = tt = 0
    for x, y in tqdm(trl, desc='Ep ' + str(ep+1) + '/10'):
        x, y = x.to(D), y.to(D)
        opt.zero_grad()
        o = m(x)
        loss = criterion(o, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0)
        opt.step()
        tc += (o.argmax(1) == y).sum().item()
        tt += y.size(0)
    m.eval()
    vc = vt = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(D), y.to(D)
            vc += (m(x).argmax(1) == y).sum().item()
            vt += y.size(0)
    ta = 100 * tc / tt
    vAcc = 100 * vc / vt
    sch.step()
    elapsed = int(time.time() - t0)
    print('Ep ' + str(ep+1) + ': Train ' + str(round(ta,1)) + '% Val ' + str(round(vAcc,1)) + '% ' + str(elapsed) + 's')
    if vAcc > best:
        best = vAcc
        torch.save(m.state_dict(), 'rec_best.pt')
        print('  Saved (' + str(round(vAcc,1)) + '%)')
print('Best: ' + str(round(best,1)) + '%')

In [ ]:
#@title Step 7: Export ONNX
m.cpu().eval()
torch.onnx.export(m, torch.randn(1, 3, 112, 112),
    'face_recognition.onnx',
    input_names=['input'], output_names=['output'], opset_version=13)
import os
sz = os.path.getsize('face_recognition.onnx') / 1e6
print('Exported: face_recognition.onnx (' + str(round(sz,1)) + 'MB)')
print('Download and place in F:/PROJECTS/NetraEdge/models/')